In [ ]:
%pip install ucimlrepo

Note: you may need to restart the kernel to use updated packages.


In [ ]:
#Jan Poreba
import pandas as pd
from ucimlrepo import fetch_ucirepo


adult = fetch_ucirepo(id=2)

# Rozdzielenie na cechy (features) i cel (targets)
X = adult.data.features
y = adult.data.targets


df = X.copy()
df['income'] = y

print(f"Początkowa liczba rekordów: {len(df)}")
print(df.head(3))
print("\n" + "="*80 + "\n")





Początkowa liczba rekordów: 48842
   age         workclass  fnlwgt  education  education-num  \
0   39         State-gov   77516  Bachelors             13   
1   50  Self-emp-not-inc   83311  Bachelors             13   
2   38           Private  215646    HS-grad              9   

       marital-status         occupation   relationship   race   sex  \
0       Never-married       Adm-clerical  Not-in-family  White  Male   
1  Married-civ-spouse    Exec-managerial        Husband  White  Male   
2            Divorced  Handlers-cleaners  Not-in-family  White  Male   

   capital-gain  capital-loss  hours-per-week native-country income  
0          2174             0              40  United-States  <=50K  
1             0             0              13  United-States  <=50K  
2             0             0              40  United-States  <=50K  




In [ ]:
# ==============================================================================
# Krok 1: Pseudonimizacja i wybór atrybutów
# Usuwamy unikalne atrybuty bezpośrednio identyfikujące (jeśli by takie były)
# i nadajemy każdemu rekordowi unikalny, losowy, anonimowy identyfikator.
# Wybieramy też nasz zbiór quasi-identyfikatorów (Q) oraz atrybut wrażliwy.
# ==============================================================================
import uuid

# Nadanie anonimowego losowego identyfikatora (pierwsze 8 znaków z UUID)
df['anon_ID'] = [str(uuid.uuid4())[:8] for _ in range(len(df))]

# quasi-identyfikatory (Q)
Q = ['age', 'education', 'sex', 'race']
# Atrybut wrażliwy
sensitive_attr = 'income'

# Tworzymy nowy zbiór roboczy zawierający tylko potrzebne kolumny
df_anon = df[['anon_ID'] + Q + [sensitive_attr]].copy()
print("Dane po dodaniu anonimowego ID (Pseudonimizacja) i zawężeniu kolumn:")
print(df_anon.head(3))
print("\n" + "="*80 + "\n")

Dane po dodaniu anonimowego ID (Pseudonimizacja) i zawężeniu kolumn:
    anon_ID  age  education   sex   race income
0  872a1bf8   39  Bachelors  Male  White  <=50K
1  9bc29a4f   50  Bachelors  Male  White  <=50K
2  b82c0b94   38    HS-grad  Male  White  <=50K




In [ ]:
# ==============================================================================
# Krok 2: Generalizacja
# Aby trudniej było zidentyfikować jednostkę, zmniejszamy precyzję danych.
# Zamiast dokładnego wieku podamy przedziały
# Edukację pogrupujemy w szersze kategorie.
# ==============================================================================
def generalize_age(age):
    if pd.isna(age): return 'Unknown'
    decade = (int(age) // 10) * 10
    return f"{decade}-{decade+9}"

def generalize_education(edu):
    higher_edu = ['Bachelors', 'Masters', 'Doctorate', 'Prof-school']
    if edu in higher_edu:
        return 'Higher Education'
    else:
        # Resztę wrzucamy do jednej grupy dla mocniejszej generalizacji
        return 'Up to College'

# Aplikujemy generalizację na zbiorze
df_anon['age'] = df_anon['age'].apply(generalize_age)
df_anon['education'] = df_anon['education'].apply(generalize_education)

print("Dane po zastosowaniu Generalizacji (wiek w przedziałach, ogólniejsze wykształcenie):")
print(df_anon.head(5))
print("\n" + "="*80 + "\n")

Dane po zastosowaniu Generalizacji (wiek w przedziałach, ogólniejsze wykształcenie):
    anon_ID    age         education     sex   race income
0  872a1bf8  30-39  Higher Education    Male  White  <=50K
1  9bc29a4f  50-59  Higher Education    Male  White  <=50K
2  b82c0b94  30-39     Up to College    Male  White  <=50K
3  319ec9e2  50-59     Up to College    Male  Black  <=50K
4  64cc91c1  20-29  Higher Education  Female  Black  <=50K




In [ ]:
# ==============================================================================
# Krok 2.5: Generowanie wszystkich klas równoważności
# ==============================================================================
def generate_equivalence_classes(dataframe, quasi_identifiers):
    """
    Generuje wszystkie klasy równoważności dla zadanego zbioru danych.
    Zwraca słownik, gdzie:
    - klucz: krotka z kombinacją wartości quasi-identyfikatorów (np. wiek, edukacja, płeć, rasa)
    - wartość: DataFrame zawierający wszystkie rekordy przypisane do tej klasy.
    """
    classes = {}
    for key, group in dataframe.groupby(quasi_identifiers):
        classes[key] = group
    return classes

# Wywołanie nowej metody na anonimizowanym zbiorze
all_eq_classes = generate_equivalence_classes(df_anon, Q)
print(f"Zidentyfikowano łącznie {len(all_eq_classes)} unikalnych klas równoważności w zbiorze przed nałożeniem k-anonimowości.")

# Wyświetlenie informacji o 3 przykładowych klasach
print("Podgląd rozmiarów 3 pierwszych klas równoważności:")
for i, (eq_key, eq_group) in enumerate(all_eq_classes.items()):
    if i >= 3:
        break
    print(f" -> Klasa Q {eq_key}: {len(eq_group)} rekordów")
print("\n" + "="*80 + "\n")

Zidentyfikowano łącznie 145 unikalnych klas równoważności w zbiorze przed nałożeniem k-anonimowości.
Podgląd rozmiarów 3 pierwszych klas równoważności:
 -> Klasa Q ('10-19', 'Higher Education', 'Female', 'Black'): 1 rekordów
 -> Klasa Q ('10-19', 'Higher Education', 'Female', 'White'): 2 rekordów
 -> Klasa Q ('10-19', 'Up to College', 'Female', 'Amer-Indian-Eskimo'): 15 rekordów




In [ ]:
# ==============================================================================
# Krok 3: Osiągnięcie k-anonimowości
# Teraz sprawdzamy klasy równoważności (EC). Klasa równoważności to grupa
# rekordów o identycznych wartościach dla wszystkich quasi-identyfikatorów Q.
# Jeśli jakaś klasa liczy mniej niż k rekordów, usuwamy te rekordy,
# aby zagwarantować, że każdy w zbiorze chowa się w tłumie liczącym co najmniej
# k identycznych osób.
# ==============================================================================
k = 10  # Ustawiamy wartość parametru k

# 1. Grupujemy po quasi-identyfikatorach i liczymy wielkość każdej klasy równoważności
ec_sizes = df_anon.groupby(Q).size().reset_index(name='ec_size')

# 2. Łączymy rozmiary klas ze zbiorem danych
df_k_anon = pd.merge(df_anon, ec_sizes, on=Q)

# 3. Odrzucamy rekordy, dla których rozmiar klasy jest mniejszy niż k
df_final = df_k_anon[df_k_anon['ec_size'] >= k].copy()

# usuwamy kolumnę pomocniczą z rozmiarem klasy
df_final = df_final.drop(columns=['ec_size'])

records_lost = len(df_anon) - len(df_final)
print(f"--- PODSUMOWANIE k-ANONIMIZACJI ---")
print(f"Ustawiono parametr k = {k}")
print(f"Liczba rekordów po anonimizacji: {len(df_final)}")
print(f"Odrzucono (supresja) {records_lost} rekordów, aby spełnić warunek k-anonimowości.\n")

# Pokażmy przykładową klasę równoważności (grupę ludzi, którzy z perspektywy adwersarza wyglądają tak samo)
sample_group = df_final[(df_final['age'] == '30-39') &
                        (df_final['sex'] == 'Female') &
                        (df_final['race'] == 'White') &
                        (df_final['education'] == 'Higher Education')]

print(f"Przykładowa klasa równoważności (Kobiety, 30-39 lat, Biała, Wyższe wykształcenie) liczy {len(sample_group)} osób.")
print("Zauważ, że anon_ID są różne i atrybut wrażliwy (income) może się różnić, ale quasi-identyfikatory są u wszystkich takie same:")
print(sample_group.head(10))

--- PODSUMOWANIE k-ANONIMIZACJI ---
Ustawiono parametr k = 10
Liczba rekordów po anonimizacji: 48667
Odrzucono (supresja) 175 rekordów, aby spełnić warunek k-anonimowości.

Przykładowa klasa równoważności (Kobiety, 30-39 lat, Biała, Wyższe wykształcenie) liczy 850 osób.
Zauważ, że anon_ID są różne i atrybut wrażliwy (income) może się różnić, ale quasi-identyfikatory są u wszystkich takie same:
      anon_ID    age         education     sex   race income
5    1501dbd3  30-39  Higher Education  Female  White  <=50K
8    306fa3b7  30-39  Higher Education  Female  White   >50K
188  b5aaa1b7  30-39  Higher Education  Female  White  <=50K
260  22c20a5a  30-39  Higher Education  Female  White  <=50K
268  ffdcc4e7  30-39  Higher Education  Female  White  <=50K
296  0117341d  30-39  Higher Education  Female  White  <=50K
422  98d0c745  30-39  Higher Education  Female  White   >50K
469  597a7e3e  30-39  Higher Education  Female  White   >50K
538  ad5fd56a  30-39  Higher Education  Female  White 